In [1]:
from pyspark.sql.functions import *
from pyspark.sql import *
import pyspark.pandas as pd
from pyspark.sql import SparkSession        
from dotenv import load_dotenv    
import os
from pathlib import Path

In [2]:
spark = SparkSession.builder.appName('ETL_TRAIL_RACES').getOrCreate()

In [52]:
path_env = Path.cwd() / '.env'
load_dotenv(path_env,override=True)

True

In [53]:
string_conexao = os.getenv("CONECTION_STRING")
string_conexao

'jdbc:sqlserver://localhost;databaseName=Soccer;user=user-pipe;password=140302;trustServerCertificate=true'

In [4]:
df_trail = (
    spark.read
    .format('csv')
    .option('header',True)
    .option('inferSchema',True)
    .option('sep',',')
    .load(os.getenv('TRAIL_RACES'))
)

In [5]:
df_trail.show()

+----------+--------------------+-----------+----------------+----------------+--------------------+--------------------+-------+--------------+--------------------+------+--------------+-------------------+
|event_date|                name|distance_km|elevation_gain_m|elevation_loss_m|             gpx_url|          event_name|country|      locality|          source_url|source|schema_version|         scraped_at|
+----------+--------------------+-----------+----------------+----------------+--------------------+--------------------+-------+--------------+--------------------+------+--------------+-------------------+
|2026-11-11|           Mons 200M|      320.0|           12800|           12800|https://tracedetr...|    Mons Ultra Trail|    BRA|   Nova Trento|https://itra.run/...|  itra|             1|2026-11-08 14:00:47|
|2026-11-11|            testrace|       56.0|              56|              56|                NULL|            AdiTest1|    FRA|            as|https://itra.run/...|  i

In [6]:
df_trail.printSchema()

root
 |-- event_date: date (nullable = true)
 |-- name: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- elevation_gain_m: integer (nullable = true)
 |-- elevation_loss_m: integer (nullable = true)
 |-- gpx_url: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- locality: string (nullable = true)
 |-- source_url: string (nullable = true)
 |-- source: string (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- scraped_at: timestamp (nullable = true)



In [7]:
df_trail.columns

['event_date',
 'name',
 'distance_km',
 'elevation_gain_m',
 'elevation_loss_m',
 'gpx_url',
 'event_name',
 'country',
 'locality',
 'source_url',
 'source',
 'schema_version',
 'scraped_at']

In [8]:
df_trail.describe().show()

+-------+-------------------+------------------+-----------------+-----------------+--------------------+--------------------+-------+--------------------+--------------------+------+--------------+
|summary|               name|       distance_km| elevation_gain_m| elevation_loss_m|             gpx_url|          event_name|country|            locality|          source_url|source|schema_version|
+-------+-------------------+------------------+-----------------+-----------------+--------------------+--------------------+-------+--------------------+--------------------+------+--------------+
|  count|               1233|              1233|             1158|             1149|                 835|                1233|   1233|                1232|                1233|  1233|          1233|
|   mean|               NULL| 90.34490673154899|3395.699481865285|3404.372497824195|                NULL|                NULL|   NULL|                NULL|                NULL|  NULL|           1.0|
| std

In [9]:
df_trail.select('name').show()

+--------------------+
|                name|
+--------------------+
|           Mons 200M|
|            testrace|
|              CNH-60|
|EVEREST TRAIL RAC...|
|Gran Trail Costa ...|
|           Mons 100M|
|           Mons 100k|
|             TGC 100|
|     THE  BRAVE 125K|
|The Last Desert A...|
|           100 Miles|
|               100km|
|               100km|
|               110km|
|2026 Inaugural Wi...|
|             35 Mile|
|             50 Mile|
|            50 Miles|
|                 50M|
|                 50k|
+--------------------+
only showing top 20 rows


In [10]:
df_trail.filter((col('country') == 'BRA') & (col('distance_km') >= 300 )).show()

+----------+---------+-----------+----------------+----------------+--------------------+----------------+-------+-----------+--------------------+------+--------------+-------------------+
|event_date|     name|distance_km|elevation_gain_m|elevation_loss_m|             gpx_url|      event_name|country|   locality|          source_url|source|schema_version|         scraped_at|
+----------+---------+-----------+----------------+----------------+--------------------+----------------+-------+-----------+--------------------+------+--------------+-------------------+
|2026-11-11|Mons 200M|      320.0|           12800|           12800|https://tracedetr...|Mons Ultra Trail|    BRA|Nova Trento|https://itra.run/...|  itra|             1|2026-11-08 14:00:47|
+----------+---------+-----------+----------------+----------------+--------------------+----------------+-------+-----------+--------------------+------+--------------+-------------------+



In [11]:
df_trail = df_trail.drop('scraped_at','gpx_url','source_url')

In [13]:
df_trail.withColumnsRenamed({
    "elevation_gain_m" :"elevation_gain",
    "elevation_loss_m" : "elevation_loss"
}).select('elevation_gain','elevation_loss').show()

+--------------+--------------+
|elevation_gain|elevation_loss|
+--------------+--------------+
|         12800|         12800|
|            56|            56|
|          2378|          2378|
|         12639|         12577|
|          5700|          5700|
|          8250|          8250|
|          6100|          6100|
|          6158|          6158|
|          6074|          6074|
|          NULL|          NULL|
|          8742|          8742|
|          3801|          3801|
|          3888|          3888|
|          1323|          1310|
|           120|           100|
|          1470|          1470|
|          1492|          1492|
|          4289|          4289|
|          2915|          2915|
|           860|           860|
+--------------+--------------+
only showing top 20 rows


In [14]:
df_trail.show()

+----------+--------------------+-----------+----------------+----------------+--------------------+-------+--------------+------+--------------+
|event_date|                name|distance_km|elevation_gain_m|elevation_loss_m|          event_name|country|      locality|source|schema_version|
+----------+--------------------+-----------+----------------+----------------+--------------------+-------+--------------+------+--------------+
|2026-11-11|           Mons 200M|      320.0|           12800|           12800|    Mons Ultra Trail|    BRA|   Nova Trento|  itra|             1|
|2026-11-11|            testrace|       56.0|              56|              56|            AdiTest1|    FRA|            as|  itra|             1|
|2026-11-13|              CNH-60|      59.13|            2378|            2378|KOLON SPORT Ultra...|    CHN|       Ninghai|  itra|             1|
|2026-11-13|EVEREST TRAIL RAC...|      185.2|           12639|           12577|  EVEREST TRAIL RACE|    NPL|  Prathvinagar| 

In [16]:
df_trail = df_trail.drop(col('schema_version'))

In [21]:
df_trail.groupBy('country').agg(
count(col('name')).alias('count_quantity_races'),
max(col('distance_km').alias('maxim_distance_race'))
).where(col('country') == 'BRA').show()

+-------+--------------------+---------------------------------------+
|country|count_quantity_races|max(distance_km AS maxim_distance_race)|
+-------+--------------------+---------------------------------------+
|    BRA|                  22|                                  320.0|
+-------+--------------------+---------------------------------------+



In [25]:
df_contagem_nulls = df_trail.select([
    sum(when(col(c).isNull(),
        1).otherwise(0)).alias(c) 
             for c in df_trail.columns
])
df_contagem_nulls.show()

+----------+----+-----------+----------------+----------------+----------+-------+--------+------+
|event_date|name|distance_km|elevation_gain_m|elevation_loss_m|event_name|country|locality|source|
+----------+----+-----------+----------------+----------------+----------+-------+--------+------+
|         0|   0|          0|              75|              84|         0|      0|       1|     0|
+----------+----+-----------+----------------+----------------+----------+-------+--------+------+



In [27]:
df_trail.filter(col('elevation_loss_m').isNull()).show()

+----------+--------------------+-----------+----------------+----------------+--------------------+-------+--------------------+------+
|event_date|                name|distance_km|elevation_gain_m|elevation_loss_m|          event_name|country|            locality|source|
+----------+--------------------+-----------+----------------+----------------+--------------------+-------+--------------------+------+
|2026-11-13|The Last Desert A...|      240.0|            NULL|            NULL|The Last Desert b...|    ARG|             Ushuaia|  itra|
|2026-11-14|The Next Loop 12 ...|      120.0|            NULL|            NULL|The Next Loop End...|    USA|               Tyler|  itra|
|2026-11-14|The Next Loop Hou...|       50.0|            NULL|            NULL|The Next Loop End...|    USA|               Tyler|  itra|
|2026-11-21|             12 Hour|      120.0|            NULL|            NULL|Salt City Fall Cl...|    USA|           Marcellus|  itra|
|2026-11-21|             12 Hour|      12

In [29]:
df_trail = df_trail.na.drop(subset=['locality'])

In [32]:
df_trail.printSchema()

root
 |-- event_date: date (nullable = true)
 |-- name: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- elevation_gain_m: integer (nullable = true)
 |-- elevation_loss_m: integer (nullable = true)
 |-- event_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- locality: string (nullable = true)
 |-- source: string (nullable = true)



In [34]:
df_trail.withColumns({
    "elevation_gain_m": col("elevation_gain_m").cast("integer"),
    "elevation_loss_m": col("elevation_loss_m").cast("integer")
}).select("elevation_gain_m","elevation_loss_m").show()

+----------------+----------------+
|elevation_gain_m|elevation_loss_m|
+----------------+----------------+
|           12800|           12800|
|              56|              56|
|            2378|            2378|
|           12639|           12577|
|            5700|            5700|
|            8250|            8250|
|            6100|            6100|
|            6158|            6158|
|            6074|            6074|
|            NULL|            NULL|
|            8742|            8742|
|            3801|            3801|
|            3888|            3888|
|            1323|            1310|
|             120|             100|
|            1470|            1470|
|            1492|            1492|
|            4289|            4289|
|            2915|            2915|
|             860|             860|
+----------------+----------------+
only showing top 20 rows
